# fasthtml

> In-kernel live inspector panel served via FastHTML

In [ ]:
#| default_exp fasthtml

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from starlette.testclient import TestClient
import paar.bridge as B, paar.fasthtml as F
from paar.fasthtml import app, _row, _broadcast
from paar.core import VarInfo

class _IP:
    user_ns={'alpha': 123}; user_ns_hidden=set()
    class events:
        @staticmethod
        def register(n, fn): pass

def test_row_renders_fields():
    html = str(_row(VarInfo(name='n', type='int', value='5', shape='3')))
    assert 'n' in html and 'int' in html and '5' in html and '3' in html
def test_rows_route_lists_vars():
    B.get_ipython = lambda: _IP()
    r = TestClient(app).get('/rows')
    assert r.status_code==200 and 'alpha' in r.text and '123' in r.text
    assert 'id="rows"' in r.text
def test_home_wires_ws_and_loader():
    r = TestClient(app).get('/')
    assert r.status_code==200
    assert 'ws-connect="/live"' in r.text and 'hx-ext="ws"' in r.text
    assert 'id="rows"' in r.text and '/rows' in r.text   # loader present
def test_broadcast_drops_dead_clients():
    F._clients.clear()
    def _bad_send(_): raise RuntimeError('dead')
    F._clients.append(('not-a-loop', _bad_send))   # scheduling against a non-loop raises -> dropped
    _broadcast('x')
    assert F._clients==[]
test_row_renders_fields(); test_rows_route_lists_vars(); test_home_wires_ws_and_loader(); test_broadcast_drops_dead_clients()

In [ ]:
#| export
import asyncio, threading
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi, HTMX
from paar.bridge import Bridge, on_change
from paar.core import VarInfo

bridge = Bridge()
app = FastHTML(exts='ws')   # bundles htmx-ext-ws from fasthtml.htmx_exts; pico on by default
_clients = []   # list[(loop, send)]
_clients_lock = threading.Lock()
_server = None

In [ ]:
#| export
def _row(v:VarInfo):
    "Render one VarInfo as a table row."
    badges = ' '.join(x for x in [v.shape and f'shape={v.shape}',
                                   v.dtype and f'dtype={v.dtype}'] if x)
    cls = 'error' if v.is_error else None
    return Tr(Td(Strong(v.name)), Td(Small(v.type, title=v.qualifier)),
              Td(Code(v.value)), Td(Small(badges)), cls=cls)

def _rows_div():
    "The rendered snapshot, wrapped in the #rows target."
    rows = [_row(v) for v in bridge.snapshot()]
    return Div(Table(Thead(Tr(Th('name'),Th('type'),Th('value'),Th('info'))),
                     Tbody(*rows)), id='rows')

def _loader(oob=False):
    "A #rows div that immediately GETs /rows (used initially and as the WS nudge)."
    kw = dict(hx_get='/rows', hx_trigger='load', hx_swap='outerHTML')
    if oob: kw['hx_swap_oob']='true'
    return Div(id='rows', **kw)

In [ ]:
#| export
@app.get('/')
def home():
    return Titled('paar', Div(_loader(), hx_ext='ws', ws_connect='/live', id='paar'))

@app.get('/rows')
def rows(): return _rows_div()

def _drop(send):
    "Remove a client by its send identity (thread-safe)."
    with _clients_lock:
        _clients[:] = [(l,s) for (l,s) in _clients if s is not send]

async def _conn(send):
    with _clients_lock: _clients.append((asyncio.get_running_loop(), send))
async def _disconn(send): _drop(send)

@app.ws('/live', conn=_conn, disconn=_disconn)
async def live(send): pass

In [ ]:
#| export
def _broadcast(fragment):
    "Push fragment to every WS client from any thread; drop clients that fail."
    with _clients_lock: targets = list(_clients)
    for loop, send in targets:
        try: fut = asyncio.run_coroutine_threadsafe(send(fragment), loop)
        except Exception: _drop(send); continue
        fut.add_done_callback(lambda f, s=send: (not f.cancelled()) and f.exception() is not None and _drop(s))

def inspector(port=8000, height=520):
    "Start the in-kernel live inspector panel and return the inline iframe."
    global _server
    if _server is None:
        _server = JupyUvi(app, port=port, daemon=True)
        on_change(lambda: _broadcast(to_xml(_loader(oob=True))))
    return HTMX(port=port, height=height, link=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()